# Case 2 — Slope Stability Monitoring: Missing Data

## Assignment context

This notebook is part of the Smart Monitoring System assignment for Master students in Civil Engineering and Territorial Protection.

**Monitoring objective:** Landslide monitoring and infrastructure safety  
**Main issue:** Missing values and communication gaps

Tasks:

1. inspect the raw sensor data;
2. detect the issue visually;
3. detect the issue statistically or with ML;
4. decide whether to correct, flag or preserve the observations;
5. prepare the dataset for ingestion, analysis, dashboarding and alerting in istSOS4Things.


## Case description

A displacement sensor monitors slope movement near a road or railway. Continuous data are important because acceleration in displacement may indicate increasing landslide risk.

The dataset contains missing values caused by communication gaps. Students should identify the gaps, evaluate their duration and reconstruct the signal only where appropriate.


## 1. Import libraries and load the dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = "dataset_2_slope_missing_data.csv"
VALUE_COL = "slope_displacement_mm"

df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
df.head()


## 2. First inspection


In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isna().sum())
print("\nSummary statistics:")
display(df.describe())


## 3. Visual inspection of raw data


In [ ]:
plt.figure(figsize=(12,4))
plt.plot(df["timestamp"], df[VALUE_COL], marker=".", linewidth=1)
plt.title(f"Raw time series: {VALUE_COL}")
plt.xlabel("Time")
plt.ylabel(VALUE_COL)
plt.grid(True)
plt.show()


## Detection and correction strategy

Suggested checks:

- count missing values;
- verify expected hourly frequency;
- plot missing values on the time series;
- compute gap length.

Possible treatment:

- linear interpolation;
- spline interpolation;
- leave long gaps as missing;
- add an imputation flag.


## 4. Visualize missing data


In [ ]:
plt.figure(figsize=(12,4))
plt.plot(df["timestamp"], df[VALUE_COL], marker=".", linewidth=1)
plt.title("Slope displacement with missing values")
plt.xlabel("Time")
plt.ylabel("Displacement [mm]")
plt.grid(True)
plt.show()

missing = df[df[VALUE_COL].isna()]
print("Number of missing observations:", len(missing))
display(missing.head())


## 5. Identify missing-data groups


In [ ]:
df["is_missing"] = df[VALUE_COL].isna()
df["missing_group"] = (df["is_missing"] != df["is_missing"].shift()).cumsum()

gap_summary = (
    df[df["is_missing"]]
    .groupby("missing_group")
    .agg(start=("timestamp", "min"), end=("timestamp", "max"), length=("timestamp", "count"))
    .reset_index(drop=True)
)
display(gap_summary)


## 6. Interpolate missing values and flag them


In [ ]:
df["slope_displacement_interpolated_mm"] = df[VALUE_COL].interpolate(method="linear")
df["quality_flag"] = np.where(df[VALUE_COL].isna(), "interpolated", "raw")

plt.figure(figsize=(12,4))
plt.plot(df["timestamp"], df[VALUE_COL], label="raw", marker=".", linewidth=1)
plt.plot(df["timestamp"], df["slope_displacement_interpolated_mm"], label="interpolated", linewidth=2)
plt.title("Raw vs interpolated displacement")
plt.xlabel("Time")
plt.ylabel("Displacement [mm]")
plt.legend()
plt.grid(True)
plt.show()

cleaned = df[["timestamp", "slope_displacement_mm", "slope_displacement_interpolated_mm", "quality_flag"]]
cleaned.to_csv("cleaned_dataset_2_slope_missing_data.csv", index=False)
cleaned.head()


## Final questions

1. What is the main data quality issue or hazardous event?
2. Which visual method was most useful?
3. Which statistical or ML method was most useful?
4. Which observations should be corrected, removed, flagged or preserved?
5. What would be a suitable alerting rule for an operational dashboard?
6. How would you model this dataset in SensorThings API?
   - Thing
   - Location
   - Sensor
   - ObservedProperty
   - Datastream
   - Observation
